In [1]:
import pathlib
import sys

_here = pathlib.Path.cwd().resolve()
for _parent in [_here, *_here.parents]:
    if (_parent / "src" / "quant_textbook").exists():
        sys.path.insert(0, str(_parent / "src"))
        break

# 53. B9 Project — SEC Filing Text & Fundamentals Forecast

> 最終成果物は「deep model」ではなく、data gate、feature lineage、baseline ladder、gradient audit、budget、nomination ruleを一つの再実行可能なevidence chainにしたもの。

## 学習目標

- pre-analysis contractをcode assertionへ変換できる
- linear baselineとNumPy MLPをdevelopment-only dataで再現できる
- point metricとcompany aggregationを分離できる
- outer access前にnominee manifestへ固定すべき項目を列挙できる
- evidence不足なら`no_model_selected`で停止できる

## 前提知識

- Week 33–36のbackprop、sequence、attention、TF–IDF
- M6 SEC data gate
- B5–B6のmodel selectionとuncertainty gate

In [2]:
import time

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

import quant_textbook as qt

pio.renderers.default = "notebook_connected"
RANDOM_SEED = 20260810
NOTEBOOK_ID = 53


def task_rng(task_id, *coordinates):
    entropy = [
        RANDOM_SEED,
        NOTEBOOK_ID,
        int(task_id),
        *(int(coordinate) for coordinate in coordinates),
    ]
    return np.random.default_rng(np.random.SeedSequence(entropy))

In [3]:
fixture = qt.load_sec_teaching_fixture()
train_mask = fixture.training_mask
validation_mask = fixture.validation_mask

assert train_mask.sum() == 192
assert validation_mask.sum() == 64
assert not np.any(fixture.target_available_dates >= np.datetime64("2023-10-23"))
assert set(fixture.partitions) == {"inner_train", "inner_validation"}

print("fixture rows:", fixture.targets.size)
print("inner train / validation:", int(train_mask.sum()), int(validation_mask.sum()))
print("numeric / sequence shape:", fixture.numeric_features.shape, fixture.token_hashes.shape)
print("locked outer rows present: False")
print("fixture hash lineage:", fixture.provenance)

fixture rows: 256
inner train / validation: 192 64
numeric / sequence shape: (256, 12) (256, 128)
locked outer rows present: False
fixture hash lineage: {'panel_artifact_sha256': '6c6008c2f28c30299e15e37613cfb0b3b22e8fd283858f5b459227c7e4a412a8', 'previous_filing_sidecar_sha256': '9ff2efef335357ff53bb1e4ba5c57f4b2e8799fc4ee5d830c55843a50026fbbc', 'normalized_manifest_sha256': '1283b9cb0992cfd2caaa942f6c869e212762c90a9abbc9a050173f5e3963daba', 'preanalysis_contract_sha256': '0aa180acbcd2b685509d6ec65fdf40f9edfcfc544ecec62c930facd0d4615b20'}


In [4]:
numeric_preprocessor = qt.fit_numeric_preprocessor(fixture.numeric_features, train_mask)
numeric_features = numeric_preprocessor.transform(fixture.numeric_features)
numeric_train = numeric_features[train_mask]
numeric_validation = numeric_features[validation_mask]
target_train = fixture.targets[train_mask]
target_validation = fixture.targets[validation_mask]
entity_validation = np.asarray(fixture.entity_ids)[validation_mask]

assert np.all(np.isfinite(numeric_features))
print("processed numeric shape:", numeric_features.shape)

processed numeric shape: (256, 24)


## 1. Frozen Project contract

| 項目 | 固定値 |
|---|---|
| target | next-quarter log Assets change |
| prediction time | previous filing availability (`known_at`) |
| inner train / validation | date cutoff 2021-01-01、company rule維持 |
| locked outer | 2023-10-23以降かつ`cik % 3 == 0` |
| primary / secondary | MAE / median absolute error |
| fixed baseline ladder | zero、pooled drift、seasonal、company mean |
| primary comparator | inner-validation MAE最小baselineを固定tie-breakで選び、outer前にfreeze |
| neural comparator | TF–IDF ridge |
| budget | 12 runs/family、200 epochs/run、100k parameters以下 |
| failure result | `no_model_selected` |

このNotebookは256行教材fixtureだけを使う。candidate教材実装はcompact inner train 192行でfitする一方、fixed baseline predictionは正式規約どおりfull 1,504-row inner training partitionだけから事前計算している。したがって同じ64-row validation上のbaseline ladder規約は検証できるが、candidateとの順位は公平なfull-data tournamentではない。full 2,195-row development searchは別artifactでtime + 2方向company-disjointの計180候補を実行済みで、all-axis gateを満たす候補はなかった。development freezeは`no_model_selected`であり、outerは開かない。

In [5]:
tfidf_model = qt.fit_hashed_tfidf(
    fixture.token_hashes, train_mask, maximum_features=256, minimum_document_frequency=2
)
tfidf = tfidf_model.transform(fixture.token_hashes)
numeric_ridge = qt.fit_sparse_ridge(numeric_train, target_train, ridge=1.0)
text_ridge = qt.fit_sparse_ridge(tfidf[train_mask], target_train, ridge=1.0)
mlp = qt.train_mlp(
    numeric_train,
    target_train,
    numeric_validation,
    target_validation,
    hidden_width=16,
    learning_rate=0.003,
    epochs=200,
    patience=20,
    l2=1e-4,
    rng=task_rng(1),
)

fixed_baseline_names = ["zero", "pooled_drift", "seasonal", "company_mean"]
project_predictions = {
    name: fixture.baseline_predictions[name][validation_mask]
    for name in fixed_baseline_names
}
project_predictions.update({
    "numeric_ridge": numeric_ridge.predict(numeric_validation),
    "hashed_tfidf_ridge": text_ridge.predict(tfidf[validation_mask]),
    "numeric_mlp": qt.mlp_predict(mlp.parameters, numeric_validation),
})
project_metrics = pd.DataFrame(
    [
        {"model": name, **qt.regression_error_table(target_validation, prediction, entity_validation)}
        for name, prediction in project_predictions.items()
    ]
).sort_values(["mae", "median_absolute_error"])
display(project_metrics)

baseline_metrics = (
    project_metrics.set_index("model").loc[fixed_baseline_names].reset_index()
)
tie_break = {name: index for index, name in enumerate(fixed_baseline_names)}
primary_baseline = min(
    fixed_baseline_names,
    key=lambda name: (
        float(baseline_metrics.loc[baseline_metrics["model"] == name, "mae"].iloc[0]),
        tie_break[name],
    ),
)
baseline_minima = {
    metric: float(baseline_metrics[metric].min())
    for metric in ("mae", "median_absolute_error", "company_macro_mae")
}
print("teaching-fixture primary baseline:", primary_baseline)
print("metric-wise fixed-baseline minima:", baseline_minima)
assert set(baseline_metrics["model"]) == set(fixed_baseline_names)

,model,mae,median_absolute_error,rmse,company_macro_mae
0,zero,0.049469,0.020475,0.114476,0.043651
1,pooled_drift,0.052109,0.024111,0.118462,0.046102
5,hashed_tfidf_ridge,0.061339,0.032573,0.124007,0.053581
3,company_mean,0.064327,0.031272,0.120754,0.055345
4,numeric_ridge,0.069652,0.027980,0.156989,0.059852
2,seasonal,0.069820,0.020475,0.159554,0.062704
6,numeric_mlp,0.129996,0.079725,0.181431,0.132689


teaching-fixture primary baseline: zero
metric-wise fixed-baseline minima: {'mae': 0.04946852858627131, 'median_absolute_error': 0.020474899964401917, 'company_macro_mae': 0.04365060333316659}


## 2. Evidence gate

教材fixtureでもfixed baseline 4本を省略しない。ただし64-row validationの順位を正式nominee選定へ転用しない。full inner validationでは、MAEをfixed baseline中の最小値から1%以上改善し、medAEとcompany-macro MAEも各metricのbaseline最小値を悪化させないことを要求する。MAE最小baselineは固定tie-breakで選び、nominee manifestと一緒にouter前にfreezeする。outerのpaired intervalはそのfrozen baselineだけを比較対象とし、outer outcomeから再選択しない。neural valueの追加主張にはTF–IDF ridgeに対する同じpoint/uncertainty gateも必要である。

この規約はteaching fixtureでzeroがpooled driftより強いと確認した後、full candidate search・nominee freeze・outer accessより前にamendmentとして記録した。元のcontract hash、観測済み情報、変更理由をcontractの \`amendments\` へ残しており、事前登録を黙って書き換えてはいない。

In [6]:
gate = pd.DataFrame(
    [
        {"artifact": "M6 panel integrity", "status": "passed"},
        {"artifact": "filing retrieval and text gate", "status": "passed"},
        {"artifact": "development-only teaching fixture", "status": "passed"},
        {"artifact": "full 2,195-row candidate search", "status": "passed: 180 development candidates"},
        {"artifact": "company-cluster paired bootstrap", "status": "not run: no nominee"},
        {"artifact": "development freeze manifest", "status": "frozen: no_model_selected"},
        {"artifact": "locked outer evaluation", "status": "unopened"},
    ]
)
modeling_decision = "no_model_selected"
assert modeling_decision == "no_model_selected"
assert gate.loc[gate["artifact"] == "locked outer evaluation", "status"].item() == "unopened"
display(gate)
print("current decision:", modeling_decision)

fig = go.Figure()
fig.add_bar(x=project_metrics["model"], y=project_metrics["mae"], name="MAE")
fig.add_bar(
    x=project_metrics["model"],
    y=project_metrics["company_macro_mae"],
    name="company macro MAE",
)
fig.update_layout(
    title="Teaching-fixture diagnostics, not nominee selection",
    yaxis_title="Absolute log-change error",
    barmode="group",
    template="plotly_white",
)
fig.show()

,artifact,status
0,M6 panel integrity,passed
1,filing retrieval and text gate,passed
2,development-only teaching fixture,passed
3,"full 2,195-row candidate search",passed: 180 development candidates
4,company-cluster paired bootstrap,not run: no nominee
5,development freeze manifest,frozen: no_model_selected
6,locked outer evaluation,unopened


current decision: no_model_selected


## 3. Deliverable checklist

正式nominee manifestには少なくとも次を固定する。

- source panel / filing sidecar / raw / normalized manifest SHA
- feature code commit、vocabulary/IDF hash、numeric scaler/imputer
- family、hyperparameters、root seedとseed offset
- parameter count、epochs、early-stopping rule、runtime environment
- inner-validation prediction hashとmetric table
- overall nominee / neural nomineeまたは`no_model_selected`
- outer access timestampと「一度だけ」の監査記録

## 4. 失敗モード

- 教材fixtureの64-row validationでnomineeを決める
- outerを見てfeature/modelを追加する
- best seedだけを報告する
- parameter budgetを超えたmodelを同じtournamentへ入れる
- row MAEだけでcompany concentrationを隠す
- neural gainが不明でもdeep-learning成功と書く

## 5. 段階別演習

### 基礎

1. Project gateの未完了artifactを列挙せよ。
2. `no_model_selected`が妥当な結論になる条件を書け。

### 標準

3. nominee manifest JSON schemaを設計せよ。
4. company-cluster paired bootstrapのresampling unitを実装せよ。

### 研究

5. outer一回評価後に許される分析と禁止するretuningをpre-commitせよ。

## 6. Exit Criteria

- [ ] SEC data/text integrity gateを通した
- [ ] full development searchと教材fixtureを区別した
- [ ] numeric/TF–IDF baselineを先に固定した
- [ ] gradient、budget、runtime、prediction hashを監査した
- [ ] outer前にnomineeまたは`no_model_selected`を凍結した
- [ ] causal/trading/representative claimをしていない

## 7. 出典

- [Goodfellow, Bengio, and Courville, *Deep Learning*](https://www.deeplearningbook.org/)
- [Glorot and Bengio (2010), Understanding the difficulty of training deep feedforward neural networks](https://proceedings.mlr.press/v9/glorot10a.html)
- [Kingma and Ba (2015), Adam](https://arxiv.org/abs/1412.6980)

- [Hochreiter and Schmidhuber (1997), Long Short-Term Memory](https://www.bioinf.jku.at/publications/older/2604.pdf)
- [Bai, Kolter, and Koltun (2018), An Empirical Evaluation of Generic Convolutional and Recurrent Networks](https://arxiv.org/abs/1803.01271)
- [Vaswani et al. (2017), Attention Is All You Need](https://arxiv.org/abs/1706.03762)

- [SEC EDGAR application programming interfaces](https://www.sec.gov/search-filings/edgar-application-programming-interfaces)
- [SEC Developer Resources](https://www.sec.gov/about/developer-resources)
- [Manning, Raghavan, and Schütze, *Introduction to Information Retrieval*](https://nlp.stanford.edu/IR-book/)